# OCO-2 GEOS CO2 Data Exploration

This notebook reads and explores the OCO-2 GEOS L3 CO2 daily data file.

In [1]:
import netCDF4 as nc
import numpy as np
import matplotlib.pyplot as plt

# Set up plotting style
plt.rcParams['figure.figsize'] = (12, 8)
plt.rcParams['font.size'] = 10

ModuleNotFoundError: No module named 'netCDF4'

## Load the NetCDF4 file

In [ ]:
# Open the NetCDF4 file
file_path = 'oco2_GEOS_L3CO2_day_20220228_B10206Ar.nc4'
dataset = nc.Dataset(file_path, 'r')

print(f"File: {file_path}")
print(f"Format: {dataset.data_model}")
print(f"\nGlobal Attributes:")
for attr in dataset.ncattrs():
    print(f"  {attr}: {dataset.getncattr(attr)}")

## Explore Dimensions

In [ ]:
print("Dimensions:")
for dim_name, dimension in dataset.dimensions.items():
    print(f"  {dim_name}: {len(dimension)} {'(unlimited)' if dimension.isunlimited() else ''}")

## Explore Variables

In [ ]:
print("Variables:")
for var_name, variable in dataset.variables.items():
    print(f"\n{var_name}:")
    print(f"  Shape: {variable.shape}")
    print(f"  Dimensions: {variable.dimensions}")
    print(f"  Data type: {variable.dtype}")
    
    # Print attributes
    if variable.ncattrs():
        print(f"  Attributes:")
        for attr in variable.ncattrs():
            print(f"    {attr}: {variable.getncattr(attr)}")

## Extract Key Variables

In [ ]:
# Extract coordinates
lats = dataset.variables['lat'][:]
lons = dataset.variables['lon'][:]

print(f"Latitude range: {lats.min():.2f} to {lats.max():.2f}")
print(f"Longitude range: {lons.min():.2f} to {lons.max():.2f}")
print(f"Grid size: {len(lats)} x {len(lons)}")

# Create meshgrid for plotting
lon_grid, lat_grid = np.meshgrid(lons, lats)

## Explore CO2 Data Variables

Let's find all CO2-related variables

In [ ]:
# Find all variables that might contain CO2 data
co2_vars = [var for var in dataset.variables.keys() if 'co2' in var.lower() or 'xco2' in var.lower()]

print("CO2-related variables:")
for var_name in co2_vars:
    var = dataset.variables[var_name]
    print(f"\n{var_name}:")
    print(f"  Shape: {var.shape}")
    print(f"  Dimensions: {var.dimensions}")
    
    # Get units if available
    if 'units' in var.ncattrs():
        print(f"  Units: {var.getncattr('units')}")
    
    # Get description if available
    if 'long_name' in var.ncattrs():
        print(f"  Description: {var.getncattr('long_name')}")

## Extract and Visualize CO2 Data

In [ ]:
# Try to find the main CO2 variable - try common names
co2_var_name = None
possible_names = ['XCO2', 'xco2', 'CO2', 'co2_column', 'XCO2_averaged']

for name in possible_names:
    if name in dataset.variables:
        co2_var_name = name
        break

if co2_var_name is None:
    # If we can't find it, just use the first CO2-related variable
    if co2_vars:
        co2_var_name = co2_vars[0]
    else:
        print("No CO2 variables found!")
        co2_var_name = None

if co2_var_name:
    print(f"Using variable: {co2_var_name}")
    co2_data = dataset.variables[co2_var_name][:]
    
    # Handle different dimensions
    if len(co2_data.shape) == 3:  # time, lat, lon
        co2_data = co2_data[0, :, :]  # Take first time step
    elif len(co2_data.shape) == 2:  # lat, lon
        pass  # Already 2D
    
    print(f"CO2 data shape: {co2_data.shape}")
    print(f"CO2 range: {np.nanmin(co2_data):.2f} to {np.nanmax(co2_data):.2f}")

## Visualize Global CO2 Distribution

In [ ]:
if co2_var_name:
    fig, ax = plt.subplots(figsize=(15, 8))
    
    # Create contour plot
    im = ax.pcolormesh(lon_grid, lat_grid, co2_data, 
                       cmap='RdYlBu_r', shading='auto')
    
    ax.set_xlabel('Longitude')
    ax.set_ylabel('Latitude')
    ax.set_title(f'Global CO2 Distribution - {co2_var_name}\n2022-02-28')
    
    # Add colorbar
    cbar = plt.colorbar(im, ax=ax)
    
    # Get units if available
    if 'units' in dataset.variables[co2_var_name].ncattrs():
        units = dataset.variables[co2_var_name].getncattr('units')
        cbar.set_label(f'CO2 ({units})')
    else:
        cbar.set_label('CO2')
    
    plt.tight_layout()
    plt.show()

## Statistics

In [ ]:
if co2_var_name:
    print("CO2 Statistics:")
    print(f"  Mean: {np.nanmean(co2_data):.4f}")
    print(f"  Median: {np.nanmedian(co2_data):.4f}")
    print(f"  Std Dev: {np.nanstd(co2_data):.4f}")
    print(f"  Min: {np.nanmin(co2_data):.4f}")
    print(f"  Max: {np.nanmax(co2_data):.4f}")
    print(f"  Valid points: {np.sum(~np.isnan(co2_data))}")
    print(f"  Missing points: {np.sum(np.isnan(co2_data))}")

## Focus on US Region

In [ ]:
if co2_var_name:
    # US bounding box
    us_west, us_east = -125, -66
    us_south, us_north = 24, 49
    
    # Find indices for US region
    lat_mask = (lats >= us_south) & (lats <= us_north)
    lon_mask = (lons >= us_west) & (lons <= us_east)
    
    # Extract US data
    us_lats = lats[lat_mask]
    us_lons = lons[lon_mask]
    us_co2 = co2_data[np.ix_(lat_mask, lon_mask)]
    
    print(f"US region: {len(us_lats)} x {len(us_lons)} grid points")
    print(f"US CO2 range: {np.nanmin(us_co2):.2f} to {np.nanmax(us_co2):.2f}")
    
    # Plot US region
    fig, ax = plt.subplots(figsize=(15, 8))
    
    us_lon_grid, us_lat_grid = np.meshgrid(us_lons, us_lats)
    im = ax.pcolormesh(us_lon_grid, us_lat_grid, us_co2, 
                       cmap='RdYlBu_r', shading='auto')
    
    ax.set_xlabel('Longitude')
    ax.set_ylabel('Latitude')
    ax.set_title(f'US CO2 Distribution - {co2_var_name}\n2022-02-28')
    ax.set_xlim(us_west, us_east)
    ax.set_ylim(us_south, us_north)
    
    cbar = plt.colorbar(im, ax=ax)
    if 'units' in dataset.variables[co2_var_name].ncattrs():
        units = dataset.variables[co2_var_name].getncattr('units')
        cbar.set_label(f'CO2 ({units})')
    else:
        cbar.set_label('CO2')
    
    plt.tight_layout()
    plt.show()

## Export Data for Visualization

Export US CO2 data in a format suitable for the particle advection visualization

In [ ]:
import json

if co2_var_name:
    # Prepare data for export
    export_data = {
        'metadata': {
            'date': '2022-02-28',
            'variable': co2_var_name,
            'units': dataset.variables[co2_var_name].getncattr('units') if 'units' in dataset.variables[co2_var_name].ncattrs() else 'unknown',
            'bounds': {
                'west': float(us_west),
                'east': float(us_east),
                'south': float(us_south),
                'north': float(us_north)
            },
            'grid_size': {
                'width': len(us_lons),
                'height': len(us_lats)
            }
        },
        'lats': us_lats.tolist(),
        'lons': us_lons.tolist(),
        'co2': np.nan_to_num(us_co2, nan=0.0).tolist()  # Replace NaN with 0
    }
    
    # Save to JSON
    with open('co2_data_us.json', 'w') as f:
        json.dump(export_data, f)
    
    print("Exported CO2 data to co2_data_us.json")

## Close the dataset

In [ ]:
dataset.close()
print("Dataset closed.")